In [2]:
!pip install optuna

In [3]:
import os
import numpy as np
import pandas as pd

In [4]:
import gdown
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error,mean_squared_error,mean_absolute_percentage_error as mape
from xgboost import XGBRegressor

tscv = TimeSeriesSplit(n_splits=3)

In [23]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
datafile_dict={
    "train_transformed_data.csv": "1w_ZOLTTc8WnLj7l8bnwo8Vuz1ii_ZBTY"
    ,"validate_transformed_data.csv" : "1SI5pS8igjgATw5fSPB8hYt5vc62aiVHF"
    ,"test_data.csv" : "1JhzqIAupBjD-vZTqnaAqL1OI9HIpD9O6"
}
#remove existing files
for filename in datafile_dict.keys():
    if os.path.exists(filename):
        os.remove(filename)
        print(f"Removed {filename}")
    else:
        print(f"{filename} not found, skipping removal.")
#doanload new files from drive
for filename, file_id in datafile_dict.items():
  gdown.download(id=file_id, output=filename, quiet=False)

train_transformed_data.csv not found, skipping removal.
validate_transformed_data.csv not found, skipping removal.
test_data.csv not found, skipping removal.


Downloading...
From: https://drive.google.com/uc?id=1w_ZOLTTc8WnLj7l8bnwo8Vuz1ii_ZBTY
To: /content/train_transformed_data.csv
100%|██████████| 28.5M/28.5M [00:00<00:00, 191MB/s]
Downloading...
From: https://drive.google.com/uc?id=1SI5pS8igjgATw5fSPB8hYt5vc62aiVHF
To: /content/validate_transformed_data.csv
100%|██████████| 6.29M/6.29M [00:00<00:00, 213MB/s]
Downloading...
From: https://drive.google.com/uc?id=1JhzqIAupBjD-vZTqnaAqL1OI9HIpD9O6
To: /content/test_data.csv
100%|██████████| 572k/572k [00:00<00:00, 66.4MB/s]


In [6]:
def load_data_dat(filename):
  if filename.endswith('.csv'):
    df=pd.read_csv(filename)
  return df

train_df= load_data_dat('train_transformed_data.csv')
val_df= load_data_dat('validate_transformed_data.csv')
test_df= load_data_dat('test_data.csv')

In [36]:
FEATURES = [
    'Store_id',#'Store_Type','Location_Type','Region_Code',
    'Store_Type_S1',
       'Store_Type_S2', 'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4',
    'Discount_Flag','Holiday',
    'day','month','weekday','weekofyear','is_weekend',
    'lag_1_#Order','lag_7_#Order','rolling_#Order_7',
    'lag_1_Sales','lag_7_Sales','rolling_Sales_7',
    'lag_1_aov',
    'Sales','#Order'
]
FEATURES_ORDERS = [
    'Store_id',#'Store_Type','Location_Type','Region_Code',
    'Store_Type_S1',
       'Store_Type_S2', 'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4',
    'Discount_Flag','Holiday',
    'day','month','weekday','weekofyear','is_weekend',
    'lag_1_#Order','lag_7_#Order','rolling_#Order_7',
    'lag_1_Sales','lag_7_Sales','rolling_Sales_7',
    'lag_1_aov'
]

FEATURES_SALES = [
    'Store_id',#'Store_Type','Location_Type','Region_Code',
    'Store_Type_S1',
       'Store_Type_S2', 'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4',
    'Discount_Flag','Holiday',
    'day','month','weekday','weekofyear','is_weekend',
    'lag_1_Sales','lag_7_Sales','rolling_Sales_7',
    'lag_1_#Order','lag_7_#Order','rolling_#Order_7',
    'lag_1_aov','pred_orders'
]

column_dtype_map_dict = {'Store_id': 'int64',
 'Store_Type_S1': 'int64',
 'Store_Type_S2': 'int64',
 'Store_Type_S3': 'int64',
 'Store_Type_S4': 'int64',
 'Location_Type_L1': 'int64',
 'Location_Type_L2': 'int64',
 'Location_Type_L3': 'int64',
 'Location_Type_L4': 'int64',
 'Location_Type_L5': 'int64',
 'Region_Code_R1': 'int64',
 'Region_Code_R2': 'int64',
 'Region_Code_R3': 'int64',
 'Region_Code_R4': 'int64',
 'Discount_Flag': 'int64',
 'Holiday': 'int64',
 'day': 'int64',
 'month': 'int64',
 'weekday': 'int64',
 'weekofyear': 'int64',
 'is_weekend': 'int64',
 'lag_1_#Order': 'float64',
 'lag_7_#Order': 'float64',
 'rolling_#Order_7': 'float64',
 'lag_1_Sales': 'float64',
 'lag_7_Sales': 'float64',
 'rolling_Sales_7': 'float64',
 'lag_1_aov': 'float64',
 'Sales': 'float64',
 '#Order': 'float64',
 'pred_orders': 'float64'}

In [8]:
train_df['Date'] = pd.to_datetime(train_df['Date'])
val_df['Date'] = pd.to_datetime(val_df['Date'])
train_df = train_df.sort_values("Date")
val_df = val_df.sort_values("Date")
train_df=train_df[FEATURES]
val_df=val_df[FEATURES]

In [ ]:
def format_df_dtype(df, features_dtype_map_dict):
    for col, dtype in features_dtype_map_dict.items():
        if col in df.columns:
            df[col] = df[col].astype(dtype)
    return df

train_df = format_df_dtype(train_df, column_dtype_map_dict)
val_df = format_df_dtype(val_df, column_dtype_map_dict)

#**XGBoost**
---

##**1. Orders**
---

In [10]:
TARGET = "#Order"  # or "#Order"

X = train_df[FEATURES_ORDERS]
y = train_df[TARGET]


In [34]:
def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(y_true) + 1e-8)

In [12]:
tscv = TimeSeriesSplit(n_splits=3)
mean_metric_scores = []
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 700),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "random_state": 42,
        "n_jobs": -1
    }

    wape_scores = []
    mae_scores = []
    mse_scores = []
    rmse_scores = []
    mape_scores = []

    for train_idx, val_idx in tscv.split(X):

        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        model = XGBRegressor(**params)
        model.fit(X_train_fold, y_train_fold)

        preds = model.predict(X_val_fold)
        #score = mean_absolute_error(y_val_fold, preds)
        #score = wape(y_val_fold, preds)

        wape_scores.append(wape(y_val_fold, preds))
        mae_scores.append(mean_absolute_error(y_val_fold, preds))
        mse_scores.append(mean_squared_error(y_val_fold, preds))
        rmse_scores.append(np.sqrt(mean_squared_error(y_val_fold, preds)))
        mape_scores.append(mape(y_val_fold, preds))

    average_metric = np.mean(mae_scores)
    mean_score = mean_metric_scores.append((np.mean(wape_scores),np.mean(mae_scores),np.mean(mse_scores),np.mean(rmse_scores),np.mean(mape_scores),params))
    return average_metric

In [13]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best Params:", study.best_params)
print("Best Score:", study.best_value)

[I 2026-04-20 18:05:31,674] A new study created in memory with name: no-name-878db11a-c49c-4e0a-9e44-0473ca5e19fe
[I 2026-04-20 18:05:56,304] Trial 0 finished with value: 12.333151021093235 and parameters: {'n_estimators': 350, 'max_depth': 9, 'learning_rate': 0.196505756458993, 'subsample': 0.7448895473837225, 'colsample_bytree': 0.7219594126069286, 'reg_alpha': 1.7876816309995114, 'reg_lambda': 0.07652898135667441}. Best is trial 0 with value: 12.333151021093235.
[I 2026-04-20 18:06:08,956] Trial 1 finished with value: 12.370100433156418 and parameters: {'n_estimators': 429, 'max_depth': 6, 'learning_rate': 0.18502235924882532, 'subsample': 0.9890994376807555, 'colsample_bytree': 0.7390206721266279, 'reg_alpha': 2.1421400969567266, 'reg_lambda': 0.8734985955192764}. Best is trial 0 with value: 12.333151021093235.
[I 2026-04-20 18:06:17,913] Trial 2 finished with value: 10.682246805124796 and parameters: {'n_estimators': 291, 'max_depth': 5, 'learning_rate': 0.03753191926462513, 'subs

Best Params: {'n_estimators': 201, 'max_depth': 10, 'learning_rate': 0.013960663099383598, 'subsample': 0.9208699091164867, 'colsample_bytree': 0.9170140235850499, 'reg_alpha': 4.824825650018285, 'reg_lambda': 2.4528319109319674}
Best Score: 10.458481578845378


In [15]:
formatted_data = []
for wape, mae, mse, rmse, mape, params in mean_metric_scores:
    # Combine metrics and the params dictionary into one flat dict
    row = {
        'WAPE': wape, 'MAE': mae, 'MSE': mse, 
        'RMSE': rmse, 'MAPE': mape
    }
    row.update(params)
    formatted_data.append(row)

df = pd.DataFrame(formatted_data)
df

,WAPE,MAE,MSE,RMSE,MAPE,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,reg_alpha,reg_lambda,random_state,n_jobs
0,0.181444,12.333151,292.817515,17.037788,4.281339e+13,350,9,0.196506,0.744890,0.721959,1.787682,0.076529,42,-1
1,0.182102,12.370100,295.421751,17.181620,4.717426e+13,429,6,0.185022,0.989099,0.739021,2.142140,0.873499,42,-1
2,0.157256,10.682247,230.545965,15.131979,4.552123e+13,291,5,0.037532,0.736128,0.770772,1.803858,4.393623,42,-1
3,0.173845,11.811471,275.506119,16.561770,4.570822e+13,327,6,0.136025,0.803454,0.829167,3.719405,0.733147,42,-1
4,0.162582,11.041894,243.750572,15.561108,4.506827e+13,424,4,0.064522,0.859433,0.721795,0.969738,1.267592,42,-1
5,0.155989,10.601301,227.064016,14.978366,4.249791e+13,225,8,0.017261,0.910620,0.966614,0.234114,0.683733,42,-1
6,0.172241,11.698949,269.760568,16.406080,4.723654e+13,343,4,0.148668,0.764540,0.952581,0.082338,3.802815,42,-1
7,0.174971,11.892488,271.683715,16.404564,4.256351e+13,428,8,0.137586,0.954579,0.808006,3.445128,0.537452,42,-1
8,0.177326,12.048521,280.112093,16.706939,4.442491e+13,519,6,0.160988,0.842962,0.843139,2.921519,2.518588,42,-1
9,0.168926,11.482187,256.779562,15.941314,4.302469e+13,592,8,0.061261,0.899320,0.811673,3.493457,0.907301,42,-1


In [ ]:
df.to_csv('xgboost_errorfile_orders.csv', header=True, index=False)

In [17]:
study.best_params

{'n_estimators': 201,
 'max_depth': 10,
 'learning_rate': 0.013960663099383598,
 'subsample': 0.9208699091164867,
 'colsample_bytree': 0.9170140235850499,
 'reg_alpha': 4.824825650018285,
 'reg_lambda': 2.4528319109319674}

##**2. Sales**
---

In [40]:
train_df= load_data_dat('train_transformed_data.csv')
val_df= load_data_dat('validate_transformed_data.csv')
test_df= load_data_dat('test_data.csv')

In [41]:
train_df['Date'] = pd.to_datetime(train_df['Date'])
val_df['Date'] = pd.to_datetime(val_df['Date'])
train_df = train_df.sort_values("Date")
val_df = val_df.sort_values("Date")
train_df=train_df[FEATURES]
val_df=val_df[FEATURES]

In [46]:
def format_df_dtype(df, features_dtype_map_dict):
    for col, dtype in features_dtype_map_dict.items():
        if col in df.columns:
            df[col] = df[col].astype(dtype)
    return df

train_df = format_df_dtype(train_df, column_dtype_map_dict)
val_df = format_df_dtype(val_df, column_dtype_map_dict)

In [43]:
TARGET = "#Order"  # or "#Order"

X = train_df[FEATURES_ORDERS]
y = train_df[TARGET]


In [44]:
order_params = {
        'n_estimators': 201,
        'max_depth': 10,
        'learning_rate': 0.013960663099383598,
        'subsample': 0.9208699091164867,
        'colsample_bytree': 0.9170140235850499,
        'reg_alpha': 4.824825650018285,
        'reg_lambda': 2.4528319109319674
 }
model_orders = XGBRegressor(**order_params)

model_orders.fit(X, y)
X['pred_orders'] = model_orders.predict(X)

In [47]:
X.head()

,Store_id,Store_Type_S1,Store_Type_S2,Store_Type_S3,Store_Type_S4,Location_Type_L1,Location_Type_L2,Location_Type_L3,Location_Type_L4,Location_Type_L5,...,weekofyear,is_weekend,lag_1_#Order,lag_7_#Order,rolling_#Order_7,lag_1_Sales,lag_7_Sales,rolling_Sales_7,lag_1_aov,pred_orders
0,1,1,0,0,0,0,0,1,0,0,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.546566
59796,152,0,1,0,0,1,0,0,0,0,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54.724590
116820,296,1,0,0,0,1,0,0,0,0,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54.487133
27324,70,1,0,0,0,1,0,0,0,0,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.832047
117216,297,0,1,0,0,0,0,1,0,0,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.585300


In [48]:
TARGET = 'Sales'
y = train_df[TARGET]

In [53]:
tscv = TimeSeriesSplit(n_splits=3)
mean_metric_scores = []
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 700),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "random_state": 42,
        "n_jobs": -1
    }

    wape_scores = []
    mae_scores = []
    mse_scores = []
    rmse_scores = []
    mape_scores = []

    for train_idx, val_idx in tscv.split(X):

        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        model = XGBRegressor(**params)
        model.fit(X_train_fold, y_train_fold)

        preds = model.predict(X_val_fold)
        #score = mean_absolute_error(y_val_fold, preds)
        #score = wape(y_val_fold, preds)

        wape_scores.append(wape(y_val_fold, preds))
        mae_scores.append(mean_absolute_error(y_val_fold, preds))
        mse_scores.append(mean_squared_error(y_val_fold, preds))
        rmse_scores.append(np.sqrt(mean_squared_error(y_val_fold, preds)))
        #mape_scores.append(mape(y_val_fold, preds))

    average_metric = np.mean(mae_scores)
    mean_score = mean_metric_scores.append((
                                np.mean(wape_scores), 
                                np.mean(mae_scores), 
                                np.mean(mse_scores), 
                                np.mean(rmse_scores), 
                                0,  # Placeholder for MAPE since it's commented out
                                params
                            ))
    return average_metric

In [54]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best Params:", study.best_params)
print("Best Score:", study.best_value)

[I 2026-04-20 18:36:29,951] A new study created in memory with name: no-name-37b272fa-ecdf-4649-8328-4f32a586cf0a
[I 2026-04-20 18:36:39,324] Trial 0 finished with value: 5386.10802150514 and parameters: {'n_estimators': 628, 'max_depth': 3, 'learning_rate': 0.13575703472513576, 'subsample': 0.990875132016154, 'colsample_bytree': 0.7957166738643743, 'reg_alpha': 2.864196070616416, 'reg_lambda': 2.4928736161292466}. Best is trial 0 with value: 5386.10802150514.
[I 2026-04-20 18:36:46,501] Trial 1 finished with value: 5235.009043998417 and parameters: {'n_estimators': 294, 'max_depth': 3, 'learning_rate': 0.018897751584733488, 'subsample': 0.7032071018245494, 'colsample_bytree': 0.9463830808167368, 'reg_alpha': 3.559519819927001, 'reg_lambda': 2.601144528095566}. Best is trial 1 with value: 5235.009043998417.
[I 2026-04-20 18:37:15,384] Trial 2 finished with value: 5325.454030882815 and parameters: {'n_estimators': 210, 'max_depth': 10, 'learning_rate': 0.06469396231641392, 'subsample': 

Best Params: {'n_estimators': 323, 'max_depth': 5, 'learning_rate': 0.026530195738347556, 'subsample': 0.7388544536271039, 'colsample_bytree': 0.8628252390092686, 'reg_alpha': 1.1499809816719329, 'reg_lambda': 4.984633296547041}
Best Score: 5099.497391305139


In [55]:
formatted_data = []
for wape, mae, mse, rmse, mape, params in mean_metric_scores:
    # Combine metrics and the params dictionary into one flat dict
    row = {
        'WAPE': wape, 'MAE': mae, 'MSE': mse, 
        'RMSE': rmse, 'MAPE': mape
    }
    row.update(params)
    formatted_data.append(row)

df = pd.DataFrame(formatted_data)
df

,WAPE,MAE,MSE,RMSE,MAPE,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,reg_alpha,reg_lambda,random_state,n_jobs
0,0.125849,5386.108022,5.255823e+07,7235.936311,0,628,3,0.135757,0.990875,0.795717,2.864196,2.492874,42,-1
1,0.122172,5235.009044,5.284561e+07,7244.371328,0,294,3,0.018898,0.703207,0.946383,3.559520,2.601145,42,-1
2,0.124417,5325.454031,5.206890e+07,7201.318061,0,210,10,0.064694,0.883933,0.984994,0.198442,0.704067,42,-1
3,0.142943,6115.378303,6.635783e+07,8121.356304,0,209,7,0.188832,0.849033,0.821762,4.771292,0.629573,42,-1
4,0.122858,5262.097789,5.114767e+07,7131.520278,0,452,6,0.031911,0.786154,0.802234,2.800268,3.664589,42,-1
5,0.128125,5486.493608,5.462399e+07,7371.392427,0,561,3,0.162937,0.805148,0.702232,0.265666,1.611689,42,-1
6,0.139685,5960.388692,6.261322e+07,7897.627862,0,360,6,0.139950,0.988104,0.792823,2.771987,4.035887,42,-1
7,0.124434,5329.379127,5.189228e+07,7184.777887,0,228,8,0.049605,0.948900,0.924277,3.495023,1.913338,42,-1
8,0.125722,5380.876954,5.276058e+07,7246.500457,0,259,8,0.067038,0.703514,0.931599,1.903776,2.214625,42,-1
9,0.122809,5255.303174,5.065923e+07,7101.575188,0,325,7,0.059200,0.756272,0.942106,4.971988,4.140518,42,-1


In [56]:
df.to_csv('xgboost_errorfile_sales.csv', header=True, index=False)

In [57]:
study.best_params

{'n_estimators': 323,
 'max_depth': 5,
 'learning_rate': 0.026530195738347556,
 'subsample': 0.7388544536271039,
 'colsample_bytree': 0.8628252390092686,
 'reg_alpha': 1.1499809816719329,
 'reg_lambda': 4.984633296547041}